# Erosión hídrica potencial y actual en la cuenca alta del río Santa (hasta La Balsa) con USLE/RUSLE
**Curso:** Hidráulica de Transporte de Sedimentos (IA8017) — PDRH-UNALM, 2026-II · Dr. Jesús Abel Mejía Marcacuzco
**Autor:** Diego Roy Jesús Aranda

Este cuaderno implementa un marco USLE/RUSLE (Wischmeier y Smith, 1978; Renard et al., 1997), con las formas de erosividad basadas en el IFM usadas en el Perú por Mejía-Marcacuzco et al. (2021), y lo aplica a una cuenca altoandina con glaciares en retroceso y pasivos mineros. Usa **Google Colab + Google Earth Engine**, las mismas herramientas que propone la Unidad 1 del sílabo.

**Elementos del marco**
1. **R**: se calculan las cuatro formulaciones basadas en el IFM (R₃, R₄, Cecchi y Sonder) para cuantificar la incertidumbre por erosividad.
2. **LS**: se comparan tres formulaciones (USLE con λ = píxel, RUSLE-McCool con λ = píxel y Moore & Burch con área de aporte).
3. **Máscara criosférica**: no se calcula pérdida de *suelo* sobre glaciares ni lagunas. Además se cuantifica el **terreno desglaciado desde ~1990** (paisaje paraglaciar).
4. **Escenario de año extremo** (por defecto 2017, Niño Costero).
5. **Hotspots**: zonas de erosión alta cruzadas con pasivos mineros y con el terreno recién desglaciado (enlace con la línea *Calidad del agua*).

> Antes de empezar necesitas una cuenta de Earth Engine registrada para **uso no comercial** (gratuito), con un *Cloud Project*. Se hace en https://code.earthengine.google.com/register y toma unos minutos.

## 0. Instalación y autenticación

In [ ]:
!pip -q install geemap
import ee, math, json, requests, io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from PIL import Image as PILImage

PROJECT_ID = 'rusle-santa-unalm'
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print('Earth Engine OK')

## 1. Parámetros del estudio
El punto de salida se ubica **aproximadamente** en la estación hidrométrica La Balsa. **Verifica las coordenadas** con la ficha oficial (SENAMHI / ANA / operador de Cañón del Pato). El código ajusta el punto al cauce de mayor área drenada en un radio de 3 km y muestra el área resultante para que la compares con la reportada en la literatura (del orden de 4 700–5 000 km²).

Si tienes la delimitación oficial de la ANA (Unidades Hidrográficas), súbela como *asset* y escribe su ruta en `BASIN_ASSET`. En ese caso se usa la delimitación oficial en lugar de HydroBASINS.

In [ ]:
# --- Dominio ---
OUTLET_LON, OUTLET_LAT = -77.83, -8.86   # aprox. La Balsa (VERIFICAR)
SNAP_RADIUS_M = 3000
BASIN_ASSET   = None   # p.ej. 'projects/TU-PROYECTO/assets/cuenca_santa_labalsa'
PASIVOS_ASSET = None   # puntos de pasivos ambientales mineros (MINEM), FeatureCollection

# --- Proyección y resolución (Santa ~77.5° O -> UTM 18S) ---
CRS, SCALE = 'EPSG:32718', 30

# --- Lluvia (CHIRPS) ---
CLIM_START, CLIM_END = 1995, 2024   # climatología de 30 años
EVENT_YEAR = 2017                   # año extremo (Niño Costero)

# --- Cobertura (Sentinel-2) ---
NDVI_START, NDVI_END = '2023-01-01', '2024-12-31'
DRY_MONTHS = (6, 8)                 # junio-agosto: mínima nieve estacional

# --- Glaciares históricos (Landsat 5) ---
GLAC_PAST_Y0, GLAC_PAST_Y1 = 1987, 1991
NDSI_THR = 0.4

# --- Factor K (Sharpley & Williams, 1990) ---
A_COEF_EPIC = 0.256   # coeficiente del término de arena (ver nota en la sección 4)

# --- LS Moore & Burch ---
AS_MAX = 1000         # tope del área específica de aporte (m) para excluir cauces

# --- Práctica de conservación ---
P_DEFAULT = 1.0       # sin inventario de prácticas de conservación

EXPORT_FOLDER = 'RUSLE_Santa'
CLASS_EDGES  = [0, 7, 10, 50, 100, 150]          # seis clases usadas en estudios peruanos (t/ha/año)
CLASS_NAMES  = ['0-7 Baja', '7-10 Ligera', '10-50 Moderada', '50-100 Media',
                '100-150 Considerable', '>150 Alta']
CLASS_COLORS = ['#8c6d31', '#b15ce6', '#f4a582', '#4daf4a', '#ff9f1c', '#d7191c']

## 2. Delimitación de la cuenca

In [ ]:
merit = ee.Image('MERIT/Hydro/v1_0_1')
pt = ee.Geometry.Point([OUTLET_LON, OUTLET_LAT])

# Ajuste del punto al cauce principal (máxima área drenada en el radio)
snap = (merit.select('upa').addBands(ee.Image.pixelLonLat())
        .reduceRegion(ee.Reducer.max(3), pt.buffer(SNAP_RADIUS_M), 90).getInfo())
snap_lon, snap_lat, snap_upa = snap['max1'], snap['max2'], snap['max']
snap_pt = ee.Geometry.Point([snap_lon, snap_lat])
print(f'Punto ajustado: {snap_lon:.4f}, {snap_lat:.4f} | área drenada MERIT = {snap_upa:,.0f} km²')

if BASIN_ASSET:
    basin = ee.FeatureCollection(BASIN_ASSET).geometry()
else:
    hb = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_12')
    box = ee.Geometry.Rectangle([-78.6, -10.4, -76.9, -8.2])
    hb_reg = hb.filterBounds(box)
    pairs = hb_reg.reduceColumns(ee.Reducer.toList(2), ['HYBAS_ID', 'NEXT_DOWN']).get('list').getInfo()
    children = {}
    for hid, nd in pairs:
        children.setdefault(int(nd), []).append(int(hid))
    start = int(hb.filterBounds(snap_pt).first().get('HYBAS_ID').getInfo())
    ids, stack = [], [start]
    while stack:
        h = stack.pop(); ids.append(h); stack.extend(children.get(h, []))
    basin = hb_reg.filter(ee.Filter.inList('HYBAS_ID', ids)).union(30).geometry()
    print(f'Subcuencas HydroBASINS nivel 12 agregadas: {len(ids)}')

basin_area = basin.area(30).divide(1e6).getInfo()
dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
el = dem.reduceRegion(ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True), basin, 90, maxPixels=1e13).getInfo()
perim = basin.perimeter(30).divide(1e3).getInfo()
print(f'Área = {basin_area:,.1f} km² | perímetro = {perim:,.1f} km')
print(f"Altitud: mín {el['elevation_min']:.0f} | media {el['elevation_mean']:.0f} | máx {el['elevation_max']:.0f} m s.n.m.")

## 3. Factor R — erosividad de la lluvia
CHIRPS (0,05°) se agrega a precipitación mensual media del periodo 1995–2024 y también para el año extremo. Con esos valores se calculan:
- **IFM** (Índice de Fournier Modificado): $IFM=\sum_{i=1}^{12} P_i^2/P$
- **R₃** (usado para USLE): $R=\sum_i 67{,}355\,(P_i^2/P)^{0{,}85}$
- **R₄** (usado para RUSLE): $R=161\,IFM^{0{,}552}$
- **Cecchi** ($4{,}67\,IFM-14{,}2$) y **Sonder** ($4{,}115\,IFM+351{,}6$).

Unidades: MJ·mm·ha⁻¹·h⁻¹·año⁻¹. CHIRPS tiene sesgos en alta montaña, así que conviene contrastarlo con estaciones SENAMHI de la cuenca (sección 11).

In [ ]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD').select('precipitation')

def monthly_precip(y0, y1):
    n = y1 - y0 + 1
    def f(m):
        m = ee.Number(m)
        col = (chirps.filter(ee.Filter.calendarRange(y0, y1, 'year'))
                     .filter(ee.Filter.calendarRange(m, m, 'month')))
        return col.sum().divide(n).rename('P').set('month', m)
    return ee.ImageCollection(ee.List.sequence(1, 12).map(f))

def r_factors(Pm):
    Pann = Pm.sum().rename('P_anual')
    IFM = Pm.map(lambda im: im.pow(2)).sum().divide(Pann).rename('IFM')
    R_eq3 = Pm.map(lambda im: im.pow(2).divide(Pann).pow(0.85).multiply(67.355)).sum().rename('R_eq3')
    R_eq4 = IFM.pow(0.552).multiply(161).rename('R_eq4')
    R_cec = IFM.multiply(4.67).subtract(14.2).rename('R_Cecchi')
    R_son = IFM.multiply(4.115).add(351.6).rename('R_Sonder')
    return ee.Image.cat([Pann, IFM, R_eq3, R_eq4, R_cec, R_son]).resample('bilinear')

Pm_clim = monthly_precip(CLIM_START, CLIM_END)
Pm_evt  = monthly_precip(EVENT_YEAR, EVENT_YEAR)
RF_clim = r_factors(Pm_clim)
RF_evt  = r_factors(Pm_evt)

def basin_stats(img, scale=1000):
    s = img.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.minMax(), '', True),
                         basin, scale, maxPixels=1e13).getInfo()
    rows = {}
    for b in img.bandNames().getInfo():
        rows[b] = {'media': s.get(f'{b}_mean'), 'mín': s.get(f'{b}_min'), 'máx': s.get(f'{b}_max')}
    return pd.DataFrame(rows).T

tab_R = pd.concat({f'Clim. {CLIM_START}-{CLIM_END}': basin_stats(RF_clim),
                   f'Año {EVENT_YEAR}': basin_stats(RF_evt)}, axis=1).round(2)
display(tab_R)

# Pluviograma mensual medio de la cuenca 
pm_vals = [ee.Image(Pm_clim.filter(ee.Filter.eq('month', m)).first())
           .reduceRegion(ee.Reducer.mean(), basin, 5000).get('P') for m in range(1, 13)]
pe_vals = [ee.Image(Pm_evt.filter(ee.Filter.eq('month', m)).first())
           .reduceRegion(ee.Reducer.mean(), basin, 5000).get('P') for m in range(1, 13)]
pm_vals, pe_vals = ee.List(pm_vals).getInfo(), ee.List(pe_vals).getInfo()
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Set','Oct','Nov','Dic']
fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(12)
ax.bar(x - 0.2, pm_vals, 0.4, label=f'Media {CLIM_START}-{CLIM_END}', color='#4c72b0')
ax.bar(x + 0.2, pe_vals, 0.4, label=f'{EVENT_YEAR}', color='#dd8452')
ax.set_xticks(x, meses); ax.set_ylabel('Precipitación (mm)'); ax.legend(frameon=False)
ax.set_title('Precipitación mensual areal (CHIRPS) — cuenca Santa hasta La Balsa')
plt.tight_layout(); plt.savefig('fig_precipitacion_mensual.png', dpi=300); plt.show()
pd.DataFrame({'mes': meses, 'P_clim_mm': pm_vals, f'P_{EVENT_YEAR}_mm': pe_vals}).to_csv('tabla_precipitacion_mensual.csv', index=False)

## 4. Factor K — erodabilidad (Sharpley & Williams, 1990; modelo EPIC)
Se usa la textura y el carbono orgánico de **SoilGrids 2.0** (0–5 cm, 250 m).

> **Nota metodológica .** En algunas versiones de la ecuación, el término de arena aparece con coeficiente 0,0256 (con arena en %). La documentación teórica de SWAT (Neitsch et al., 2011) y Williams (1995) usan **0,256**. Con 0,256 K queda en el rango 0,015–0,017 reportado para suelos peruanos, por eso es el valor por defecto. Con 0,0256 se obtiene el ejemplo de la diapositiva (K = 0,0455). El parámetro `A_COEF_EPIC` permite reportar ambos.

In [ ]:
def sg(v):
    return ee.Image(f'projects/soilgrids-isric/{v}_mean').select(f'{v}_0-5cm_mean')

SAN = sg('sand').divide(10)    # g/kg -> %
SIL = sg('silt').divide(10)
CLA = sg('clay').divide(10)
OC  = sg('soc').divide(100)    # dg/kg -> %
SN1 = SAN.divide(100).multiply(-1).add(1)

def k_epic(a_coef):
    w1 = SAN.multiply(-a_coef).multiply(SIL.divide(100).multiply(-1).add(1)).exp().multiply(0.3).add(0.2)
    w2 = SIL.divide(CLA.add(SIL)).pow(0.3)
    w3 = OC.multiply(0.25).divide(OC.add(OC.multiply(-2.95).add(3.72).exp())).multiply(-1).add(1)
    w4 = SN1.multiply(0.7).divide(SN1.add(SN1.multiply(22.9).add(-5.51).exp())).multiply(-1).add(1)
    return w1.multiply(w2).multiply(w3).multiply(w4).multiply(0.1317)

K = k_epic(A_COEF_EPIC).rename('K')
K_alt = k_epic(0.0256).rename('K_coef0256')
display(basin_stats(ee.Image.cat([K, K_alt, SAN.rename('arena'), SIL.rename('limo'),
                                  CLA.rename('arcilla'), OC.rename('C_org')]), 250).round(4))

## 5. Factor LS — topografía (tres formulaciones)
DEM: **NASADEM** (30 m). ALOS PALSAR 12,5 m no está en Earth Engine; puede usarse en una versión QGIS.
- **USLE (Wischmeier y Smith, 1978):** λ = tamaño de píxel, *m* según la Tabla 6, $S=0{,}065+0{,}045s+0{,}0065s^2$.
- **RUSLE (McCool et al., 1987):** $m=\beta/(1+\beta)$ y S de McCool (umbral de 9 %).
- **Moore & Burch (1986):** $LS=(A_s/22{,}13)^{0{,}4}(\sin\beta/0{,}0896)^{1{,}3}$, con $A_s$ obtenido del área drenada MERIT. La diapositiva omite el exponente 0,4 (0,4–0,6 en la literatura). Aquí se usa 0,4.

In [ ]:
slope_deg = ee.Terrain.slope(dem)
th   = slope_deg.multiply(math.pi / 180)
sinth = th.sin()
s_pct = th.tan().multiply(100)
lam = SCALE

# USLE (Wischmeier & Smith)
m_usle = ee.Image(0.2).where(s_pct.gte(1), 0.3).where(s_pct.gte(3), 0.4).where(s_pct.gte(5), 0.5)
L_usle = ee.Image(lam / 22.13).pow(m_usle)
S_usle = s_pct.pow(2).multiply(0.0065).add(s_pct.multiply(0.045)).add(0.065)
LS_usle = L_usle.multiply(S_usle).rename('LS_USLE')

# RUSLE (McCool et al.)
beta = sinth.divide(0.0896).divide(sinth.pow(0.8).multiply(3).add(0.56))
m_r  = beta.divide(beta.add(1))
L_r  = ee.Image(lam / 22.13).pow(m_r)
S_r  = sinth.multiply(10.8).add(0.03).where(s_pct.gte(9), sinth.multiply(16.8).subtract(0.5))
LS_rusle = L_r.multiply(S_r).rename('LS_RUSLE')

# Moore & Burch con área específica de aporte
As = merit.select('upa').multiply(1e6).divide(90).min(AS_MAX)
LS_mb = As.divide(22.13).pow(0.4).multiply(sinth.divide(0.0896).pow(1.3)).rename('LS_MooreBurch')

tab_slope = basin_stats(ee.Image.cat([slope_deg.rename('pendiente_grados'), s_pct.rename('pendiente_%'),
                                      LS_usle, LS_rusle, LS_mb]), 90).round(2)
display(tab_slope)

## 6. Factor C, máscara criosférica y terreno desglaciado
- **C (Van der Knijff et al., 2000):** $C=\exp[-2\,NDVI/(1-NDVI)]$ con NDVI mediano de Sentinel-2 (2023–2024).
- **C por tabla (sensibilidad):** valores de tabla asignados a ESA WorldCover. Pajonal = 0,10, porque en la puna hay sobrepastoreo frecuente.
- **Máscara:** glaciar (NDSI ≥ 0,4 en época seca), nieve permanente y agua. Ahí no hay *suelo* que perder.
- **Desglaciado desde ~1990:** era glaciar en Landsat 5 (1987–1991, época seca) y ya no lo es en Sentinel-2. Es el material paraglaciar recién expuesto: morrenas y sedimento suelto que alimenta los finos del río.

In [ ]:
def s2_prep(im):
    scl = im.select('SCL')
    ok = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return ee.Image(im.select(['B3', 'B4', 'B8', 'B11']).divide(10000).updateMask(ok).copyProperties(im, ['system:time_start']))

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(basin)
      .filterDate(NDVI_START, NDVI_END).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
      .map(s2_prep))
s2_all = s2.median()
s2_dry = s2.filter(ee.Filter.calendarRange(DRY_MONTHS[0], DRY_MONTHS[1], 'month')).median()
NDVI = s2_all.normalizedDifference(['B8', 'B4']).rename('NDVI')
NDSI_now = s2_dry.normalizedDifference(['B3', 'B11']).rename('NDSI')

C_ndvi = NDVI.expression('exp(-2 * n / (1 - n))', {'n': NDVI.max(0).min(0.99)}).min(1).rename('C_NDVI')

wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
C_tab = (wc.remap([10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
                  [0.003, 0.03, 0.10, 0.30, 0.08, 0.90, 0, 0, 0.01, 0.01, 0.05])
           .rename('C_tabla'))

# Glaciar actual y agua
glacier_now = NDSI_now.gte(NDSI_THR).And(s2_dry.select('B4').gt(0.2))   # evita confundir agua con hielo
water = wc.eq(80).Or(NDVI.lt(0).And(NDSI_now.lt(NDSI_THR)))

# Glaciar ~1990 (Landsat 5 C2 L2)
def l5_prep(im):
    qa = im.select('QA_PIXEL')
    ok = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    sr = im.select(['SR_B2', 'SR_B5']).multiply(0.0000275).add(-0.2)
    return sr.updateMask(ok)

l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterBounds(basin)
      .filter(ee.Filter.calendarRange(GLAC_PAST_Y0, GLAC_PAST_Y1, 'year'))
      .filter(ee.Filter.calendarRange(DRY_MONTHS[0] - 1, DRY_MONTHS[1] + 1, 'month'))
      .map(l5_prep).median())
NDSI_past = l5.normalizedDifference(['SR_B2', 'SR_B5'])
glacier_past = NDSI_past.gte(NDSI_THR)
deglaciated = glacier_past.And(glacier_now.Not()).And(water.Not()).rename('desglaciado')

nomask = glacier_now.Or(water).Or(wc.eq(70)).Not()     # 1 = se calcula erosión
P = ee.Image(P_DEFAULT).rename('P')

pa = ee.Image.pixelArea().divide(1e6)
areas = ee.Image.cat([pa.updateMask(glacier_past).rename('glaciar_1990'),
                      pa.updateMask(glacier_now).rename('glaciar_actual'),
                      pa.updateMask(deglaciated).rename('desglaciado'),
                      pa.updateMask(water).rename('agua')]).reduceRegion(
    ee.Reducer.sum(), basin, SCALE, crs=CRS, maxPixels=1e13, tileScale=4).getInfo()
display(pd.Series(areas, name='km²').round(2))

## 7. Pérdida de suelo A (t/ha/año)
Definiciones:
- **Erosión potencial** = $R\cdot K\cdot LS$ (sin cobertura).
- **Erosión potencial actual** = $R\cdot K\cdot LS\cdot C\cdot P$.

Combinaciones: USLE (R ec. 3 + LS_USLE), RUSLE (R ec. 4 + LS_RUSLE) y la variante **RUSLE-MB** (R ec. 4 + Moore & Burch), que se propone como la más física para cuencas extensas.

In [ ]:
def soil_loss(RF):
    R3, R4 = RF.select('R_eq3'), RF.select('R_eq4')
    out = ee.Image.cat([
        R3.multiply(K).multiply(LS_usle).rename('Apot_USLE'),
        R4.multiply(K).multiply(LS_rusle).rename('Apot_RUSLE'),
        R3.multiply(K).multiply(LS_usle).multiply(C_ndvi).multiply(P).rename('Aact_USLE'),
        R4.multiply(K).multiply(LS_rusle).multiply(C_ndvi).multiply(P).rename('Aact_RUSLE'),
        R4.multiply(K).multiply(LS_mb).multiply(C_ndvi).multiply(P).rename('Aact_RUSLE_MB'),
        R4.multiply(K).multiply(LS_rusle).multiply(C_tab).multiply(P).rename('Aact_RUSLE_Ctabla'),
    ])
    return out.updateMask(nomask).clip(basin)

A_clim = soil_loss(RF_clim)
A_evt  = soil_loss(RF_evt)

def class_image(img):
    return (img.gte(7).add(img.gte(10)).add(img.gte(50)).add(img.gte(100)).add(img.gte(150))).rename('cls')

def class_table(img):
    grp = pa.addBands(class_image(img)).updateMask(img.mask()).reduceRegion(
        ee.Reducer.sum().group(groupField=1, groupName='cls'), basin, SCALE, crs=CRS,
        maxPixels=1e13, tileScale=8).get('groups').getInfo()
    d = {g['cls']: g['sum'] for g in grp}
    tot = sum(d.values())
    return pd.DataFrame({'Rango (t/ha/año)': CLASS_NAMES,
                         'Área (km²)': [d.get(i, 0) for i in range(6)],
                         'Porcentaje (%)': [100 * d.get(i, 0) / tot for i in range(6)]})

def summary(Aimg):
    t_ha = Aimg.multiply(ee.Image.pixelArea().divide(1e4))   # t/año por píxel
    s1 = Aimg.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.median(), '', True), basin, SCALE,
                           crs=CRS, maxPixels=1e13, tileScale=8).getInfo()
    s2 = t_ha.reduceRegion(ee.Reducer.sum(), basin, SCALE, crs=CRS, maxPixels=1e13, tileScale=8).getInfo()
    rows = {}
    for b in Aimg.bandNames().getInfo():
        rows[b] = {'A media (t/ha/año)': s1[f'{b}_mean'], 'A mediana (t/ha/año)': s1[f'{b}_median'],
                   'Pérdida total (Mt/año)': s2[b] / 1e6}
    return pd.DataFrame(rows).T

res_clim = summary(A_clim); res_evt = summary(A_evt)
tab_resumen = pd.concat({f'Clim. {CLIM_START}-{CLIM_END}': res_clim, f'Año {EVENT_YEAR}': res_evt}, axis=1).round(2)
display(tab_resumen); tab_resumen.to_csv('tabla_resumen_A.csv')

tablas_clases = {}
for b in ['Apot_USLE', 'Apot_RUSLE', 'Aact_USLE', 'Aact_RUSLE', 'Aact_RUSLE_MB']:
    t = class_table(A_clim.select(b)).round(2); tablas_clases[b] = t
    print(f'\n=== {b} (clim.) ==='); display(t); t.to_csv(f'tabla_clases_{b}.csv', index=False)

## 8. Sensibilidad: ¿cuánto pesa la elección de R y de LS?
Para un revisor Q2/Q3 este es el aporte metodológico central: la dispersión del resultado según la ecuación elegida.

In [ ]:
sens = {}
for rname in ['R_eq3', 'R_eq4', 'R_Cecchi', 'R_Sonder']:
    for lsname, ls in [('LS_USLE', LS_usle), ('LS_RUSLE', LS_rusle), ('LS_MooreBurch', LS_mb)]:
        sens[f'{rname} × {lsname}'] = (RF_clim.select(rname).multiply(K).multiply(ls)
                                        .multiply(C_ndvi).multiply(P).updateMask(nomask).rename('A'))
img_sens = ee.Image.cat([v.rename(k.replace(' × ', '__')) for k, v in sens.items()])
st = img_sens.reduceRegion(ee.Reducer.mean(), basin, 90, crs=CRS, maxPixels=1e13, tileScale=8).getInfo()
tab_sens = pd.DataFrame([{'R': k.split('__')[0], 'LS': k.split('__')[1], 'A media (t/ha/año)': v}
                         for k, v in st.items()]).pivot(index='R', columns='LS', values='A media (t/ha/año)').round(2)
display(tab_sens); tab_sens.to_csv('tabla_sensibilidad_R_LS.csv')
cv = 100 * np.nanstd(tab_sens.values) / np.nanmean(tab_sens.values)
print(f'Coeficiente de variación entre formulaciones: {cv:.1f} %')

## 9. Lectura por pisos altitudinales, terreno desglaciado y pasivos mineros
Aquí el modelo empieza a hablar de personas: en qué piso se concentra la erosión, cuánto aporta el terreno que dejó el hielo y qué pasivos mineros están dentro de zonas de erosión alta, aguas arriba de bocatomas y parcelas.

In [ ]:
A_main = A_clim.select('Aact_RUSLE_MB')
bands = ee.Image(0).where(dem.gte(3000), 1).where(dem.gte(4000), 2).where(dem.gte(4800), 3).rename('piso')
labels = ['< 3000 m', '3000-4000 m', '4000-4800 m', '> 4800 m']
t_px = A_main.multiply(ee.Image.pixelArea().divide(1e4))
g = (pa.addBands(t_px).addBands(bands).updateMask(A_main.mask())
     .reduceRegion(ee.Reducer.sum().repeat(2).group(groupField=2, groupName='piso'),
                   basin, SCALE, crs=CRS, maxPixels=1e13, tileScale=8).get('groups').getInfo())
rows = []
for x in g:
    area_km2, t_yr = x['sum']
    rows.append({'Piso': labels[int(x['piso'])], 'Área (km²)': area_km2,
                 'Pérdida (Mt/año)': t_yr / 1e6, 'Aporte (%)': None,
                 'A media (t/ha/año)': t_yr / (area_km2 * 100)})
tab_pisos = pd.DataFrame(rows)
tab_pisos['Aporte (%)'] = 100 * tab_pisos['Pérdida (Mt/año)'] / tab_pisos['Pérdida (Mt/año)'].sum()
display(tab_pisos.round(2)); tab_pisos.to_csv('tabla_pisos_altitudinales.csv', index=False)

# Terreno desglaciado
dg = (pa.addBands(t_px).updateMask(deglaciated).updateMask(A_main.mask())
      .reduceRegion(ee.Reducer.sum().repeat(2), basin, SCALE, crs=CRS, maxPixels=1e13, tileScale=8)
      .get('sum').getInfo())
tot_t = tab_pisos['Pérdida (Mt/año)'].sum()
print(f'Terreno desglaciado con suelo modelable: {dg[0]:.1f} km² | pérdida {dg[1]/1e6:.3f} Mt/año '
      f'({100*dg[1]/1e6/tot_t:.1f} % del total, en {100*dg[0]/basin_area:.2f} % del área)')

# Hotspots
hot = A_main.gte(50)
hot_area = pa.updateMask(hot).reduceRegion(ee.Reducer.sum(), basin, SCALE, crs=CRS,
                                           maxPixels=1e13, tileScale=8).getInfo()
print(f"Hotspots (A ≥ 50 t/ha/año): {list(hot_area.values())[0]:.1f} km²")

if PASIVOS_ASSET:
    pas = ee.FeatureCollection(PASIVOS_ASSET).filterBounds(basin)
    def near(f):
        buf = f.geometry().buffer(1000)
        st = ee.Image.cat([A_main.rename('A'), hot.rename('hot')]).reduceRegion(
            ee.Reducer.mean(), buf, SCALE, crs=CRS, maxPixels=1e9)
        return f.set({'A_media_1km': st.get('A'), 'frac_hotspot_1km': st.get('hot')})
    feats = pas.map(near).getInfo()['features']
    tab_pas = pd.DataFrame([f['properties'] for f in feats]).sort_values('A_media_1km', ascending=False)
    display(tab_pas.head(20)); tab_pas.to_csv('tabla_pasivos_erosion.csv', index=False)
else:
    print('Sin PASIVOS_ASSET: sube el inventario de pasivos mineros (MINEM) para el cruce.')

## 10. Producción de sedimentos a la salida (orden de magnitud)
No toda la erosión de ladera llega al río. Con la razón de entrega de sedimentos de **Vanoni (1975)**, $SDR = 0{,}4724\,A^{-0{,}125}$ (A en km²), se estima un rendimiento comparable con los registros de La Balsa o con estudios publicados. Este es el puente entre la erosión de ladera y el transporte fluvial que sigue en el curso.

In [ ]:
SDR = 0.4724 * basin_area ** (-0.125)
for b in ['Aact_USLE', 'Aact_RUSLE', 'Aact_RUSLE_MB']:
    Mt = res_clim.loc[b, 'Pérdida total (Mt/año)']
    print(f'{b}: erosión {Mt:.2f} Mt/año × SDR {SDR:.3f} = {Mt*SDR:.2f} Mt/año '
          f'≈ {Mt*SDR*1e6/basin_area:,.0f} t/km²/año')

## 11. Mapas (interactivo y figuras para el paper)

In [ ]:
import geemap
m = geemap.Map(center=[snap_lat + 0.6, snap_lon + 0.3], zoom=9)
m.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(basin)]), 1, 2), {'palette': ['black']}, 'Cuenca')
m.addLayer(RF_clim.select('R_eq4').clip(basin), {'min': 0, 'max': 3000, 'palette': ['#ffffcc', '#41b6c4', '#253494']}, 'R ec.4')
m.addLayer(K.clip(basin), {'min': 0.01, 'max': 0.04, 'palette': ['#ffffd4', '#fe9929', '#993404']}, 'K')
m.addLayer(LS_mb.clip(basin), {'min': 0, 'max': 20, 'palette': ['#f7f7f7', '#636363', '#000000']}, 'LS Moore-Burch')
m.addLayer(C_ndvi.clip(basin), {'min': 0, 'max': 1, 'palette': ['#1a9850', '#fee08b', '#d73027']}, 'C (NDVI)')
m.addLayer(class_image(A_main), {'min': 0, 'max': 5, 'palette': [c[1:] for c in CLASS_COLORS]}, 'Erosión actual RUSLE-MB')
m.addLayer(glacier_now.selfMask().clip(basin), {'palette': ['#00bfff']}, 'Glaciar actual')
m.addLayer(deglaciated.selfMask().clip(basin), {'palette': ['#ff00ff']}, 'Desglaciado desde ~1990')
m.addLayer(snap_pt, {'color': 'red'}, 'Salida (La Balsa)')
m

In [ ]:
outline = ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(basin)]), 1, 2).visualize(palette=['000000'])
region = basin.bounds(1000)

def thumb(vis_img):
    url = vis_img.blend(outline).getThumbURL({'region': region, 'dimensions': 1400, 'crs': CRS, 'format': 'png'})
    return PILImage.open(io.BytesIO(requests.get(url).content))

def fig_cont(img, vmin, vmax, palette, title, label, fname):
    pic = thumb(img.clip(basin).visualize(min=vmin, max=vmax, palette=palette))
    fig, ax = plt.subplots(figsize=(6.5, 8)); ax.imshow(pic); ax.axis('off'); ax.set_title(title, fontsize=11)
    cm = ListedColormap(palette) if len(palette) > 6 else LinearSegmentedColormap.from_list('p', palette)
    cb = fig.colorbar(ScalarMappable(Normalize(vmin, vmax), cm), ax=ax, fraction=0.035, pad=0.02); cb.set_label(label)
    plt.tight_layout(); plt.savefig(fname, dpi=300); plt.show()

def fig_class(img, title, fname):
    pic = thumb(class_image(img).visualize(min=0, max=5, palette=CLASS_COLORS))
    fig, ax = plt.subplots(figsize=(6.5, 8)); ax.imshow(pic); ax.axis('off'); ax.set_title(title, fontsize=11)
    ax.legend(handles=[mpatches.Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)],
              title='t/ha/año', loc='lower left', fontsize=8, frameon=True)
    plt.tight_layout(); plt.savefig(fname, dpi=300); plt.show()

fig_cont(RF_clim.select('R_eq4'), 0, 3000, ['#ffffcc', '#41b6c4', '#253494'], 'Factor R (ec. 4)', 'MJ·mm/ha·h·año', 'fig_R.png')
fig_cont(K, 0.01, 0.04, ['#ffffd4', '#fe9929', '#993404'], 'Factor K (Sharpley & Williams)', 't·ha·h/ha·MJ·mm', 'fig_K.png')
fig_cont(LS_mb, 0, 20, ['#f7f7f7', '#636363', '#000000'], 'Factor LS (Moore & Burch)', 'adimensional', 'fig_LS.png')
fig_cont(C_ndvi, 0, 1, ['#1a9850', '#fee08b', '#d73027'], 'Factor C (NDVI Sentinel-2)', 'adimensional', 'fig_C.png')
fig_class(A_clim.select('Apot_RUSLE'), 'Erosión potencial RUSLE (R·K·LS)', 'fig_Apot_RUSLE.png')
fig_class(A_clim.select('Aact_USLE'), 'Erosión potencial actual USLE', 'fig_Aact_USLE.png')
fig_class(A_clim.select('Aact_RUSLE'), 'Erosión potencial actual RUSLE', 'fig_Aact_RUSLE.png')
fig_class(A_main, 'Erosión potencial actual RUSLE-MB (climatología)', 'fig_Aact_RUSLE_MB.png')
fig_class(A_evt.select('Aact_RUSLE_MB'), f'Erosión potencial actual RUSLE-MB ({EVENT_YEAR})', 'fig_Aact_RUSLE_MB_evento.png')

pic = thumb(ee.Image(0).visualize(palette=['ffffff']).clip(basin)
            .blend(glacier_now.selfMask().visualize(palette=['00bfff']))
            .blend(deglaciated.selfMask().visualize(palette=['ff00ff'])))
fig, ax = plt.subplots(figsize=(6.5, 8)); ax.imshow(pic); ax.axis('off')
ax.set_title('Glaciar actual y terreno desglaciado desde ~1990')
ax.legend(handles=[mpatches.Patch(color='#00bfff', label='Glaciar actual (S2)'),
                   mpatches.Patch(color='#ff00ff', label='Desglaciado (L5 1987-91 → S2)')], loc='lower left')
plt.tight_layout(); plt.savefig('fig_glaciares.png', dpi=300); plt.show()

## 12. Exportación (GeoTIFF a Google Drive para QGIS) y descarga de tablas y figuras

In [ ]:
to_export = {'R_eq3': RF_clim.select('R_eq3'), 'R_eq4': RF_clim.select('R_eq4'), 'K': K,
             'LS_USLE': LS_usle, 'LS_RUSLE': LS_rusle, 'LS_MooreBurch': LS_mb, 'C_NDVI': C_ndvi,
             'Aact_USLE': A_clim.select('Aact_USLE'), 'Aact_RUSLE': A_clim.select('Aact_RUSLE'),
             'Aact_RUSLE_MB': A_main, f'Aact_RUSLE_MB_{EVENT_YEAR}': A_evt.select('Aact_RUSLE_MB'),
             'desglaciado': deglaciated.selfMask().toFloat()}
for name, img in to_export.items():
    ee.batch.Export.image.toDrive(image=img.toFloat().clip(basin), description=f'Santa_{name}',
        folder=EXPORT_FOLDER, region=basin, scale=SCALE, crs=CRS, maxPixels=1e13).start()
ee.batch.Export.table.toDrive(ee.FeatureCollection([ee.Feature(basin, {'area_km2': basin_area})]),
    description='Santa_cuenca_limite', folder=EXPORT_FOLDER, fileFormat='SHP').start()
print('Exportaciones enviadas: revisa la pestaña Tasks en https://code.earthengine.google.com')

import glob, zipfile
with zipfile.ZipFile('resultados_RUSLE_Santa.zip', 'w') as z:
    for f in glob.glob('*.png') + glob.glob('*.csv'):
        z.write(f)
tab_R.to_csv('tabla_factor_R.csv')
try:
    from google.colab import files; files.download('resultados_RUSLE_Santa.zip')
except Exception:
    print('Descarga manual: resultados_RUSLE_Santa.zip')

## 13. Qué enviar para completar el paper y la presentación
Sube el archivo **resultados_RUSLE_Santa.zip** (tablas CSV y figuras PNG) a la conversación. Con eso se completan los marcadores `[RESULTADO]` del paper y de las diapositivas.

**Límites que el paper debe declarar con honestidad:** CHIRPS en alta montaña, SoilGrids a 250 m, la USLE/RUSLE no representa erosión en cárcavas, flujos de detritos ni aportes glaciares subglaciales (que dominan los finos del Santa), y el SDR es empírico. Por eso los resultados son un **indicador espacial de riesgo**, no un balance de sedimentos cerrado.

## 14. Pasivos ambientales mineros (MINEM vía Geocatmin del INGEMMET) y validación con la estación Santa
Se consulta el servicio público ArcGIS del INGEMMET, se calcula la erosión en un radio de 500 m de cada pasivo y se compara con la cuenca. El hotspot se define como el 10 % del área con mayor erosión RUSLE. Luego se contrasta el rendimiento estimado con el medido por Morera et al. (2013).

In [ ]:
import requests, pandas as pd
URL = 'https://geocatminnube.ingemmet.gob.pe/arcgis/rest/services/SERV_PASIVO_AMBIENTAL/MapServer/0/query'
bb = basin.bounds().coordinates().getInfo()[0]
xs = [p[0] for p in bb]; ys = [p[1] for p in bb]
feats, off = [], 0
while True:
    r = requests.get(URL, params={'where': '1=1', 'geometry': f'{min(xs)},{min(ys)},{max(xs)},{max(ys)}',
        'geometryType': 'esriGeometryEnvelope', 'inSR': 4326, 'spatialRel': 'esriSpatialRelIntersects',
        'outFields': 'objectid,exunidad,tipo,subtipo,provincia,distrito,cuenca', 'outSR': 4326,
        'f': 'geojson', 'resultOffset': off, 'resultRecordCount': 1000}, timeout=120).json()
    fs = r.get('features', [])
    feats += fs
    if len(fs) < 1000: break
    off += 1000
print('PAM en el rectángulo:', len(feats))
for f in feats:
    f['properties'] = {k: ('' if v is None else v) for k, v in f['properties'].items()}
pam = ee.FeatureCollection([ee.Feature(ee.Geometry.Point(f['geometry']['coordinates']), f['properties']) for f in feats if f.get('geometry')]).filterBounds(basin)
n_pam = pam.size().getInfo()
print('PAM dentro de la cuenca:', n_pam)
img = ee.Image.cat([A_main.rename('A'), A_main.gte(50).rename('hot')])
stats = img.reduceRegions(pam.map(lambda f: f.buffer(500)), ee.Reducer.mean(), 30, crs=CRS, tileScale=8)
rows = [f['properties'] for f in stats.getInfo()['features']]
tp = pd.DataFrame(rows)
tp['A'] = pd.to_numeric(tp.get('A'), errors='coerce'); tp['hot'] = pd.to_numeric(tp.get('hot'), errors='coerce')
A_bas = float(res_clim.loc['Aact_RUSLE_MB', 'A media (t/ha/año)'])
v = tp.dropna(subset=['A'])
print(f'PAM con suelo modelable en 500 m: {len(v)} de {len(tp)}')
print(f'A media en entorno de PAM: {v.A.mean():.1f} t/ha/año (mediana {v.A.median():.1f}) vs cuenca {A_bas:.1f}')
print(f'PAM con >20 % de su entorno en hotspot: {(v.hot>0.2).sum()} ({100*(v.hot>0.2).mean():.1f} %)')
print(f'PAM con >50 % de su entorno en hotspot: {(v.hot>0.5).sum()} ({100*(v.hot>0.5).mean():.1f} %)')
print('--- por tipo ---')
print(v.groupby('tipo').agg(n=('A','size'), A_media=('A','mean'), frac_hot=('hot','mean')).sort_values('n', ascending=False).round(2).to_string())
print('--- por provincia ---')
print(v.groupby('provincia').agg(n=('A','size'), A_media=('A','mean'), frac_hot=('hot','mean')).sort_values('n', ascending=False).round(2).to_string())
print('--- ex unidades con mas PAM en hotspot (>50 %) ---')
print(v[v.hot>0.5].groupby('exunidad').agg(n=('A','size'), A_media=('A','mean')).sort_values('n', ascending=False).head(12).round(1).to_string())
tp.to_csv('tabla_pasivos_erosion.csv', index=False)
pam_pts = pam
print('PRECIP_CLIM', [round(x,1) for x in pm_vals]); print('PRECIP_2017', [round(x,1) for x in pe_vals])


In [ ]:
AR = A_clim.select('Aact_RUSLE')
p = AR.reduceRegion(ee.Reducer.percentile([50, 90]), basin, 90, crs=CRS, maxPixels=1e13, tileScale=8).getInfo()
P90 = p['Aact_RUSLE_p90']; print(f'Umbral hotspot (P90 RUSLE): {P90:.1f} t/ha/año')
img2 = ee.Image.cat([AR.rename('A'), AR.gte(P90).rename('hot')])
st2 = img2.reduceRegions(pam.map(lambda f: f.buffer(500)), ee.Reducer.mean(), 30, crs=CRS, tileScale=8)
t2 = pd.DataFrame([f['properties'] for f in st2.getInfo()['features']])
t2['A'] = pd.to_numeric(t2['A'], errors='coerce'); t2['hot'] = pd.to_numeric(t2['hot'], errors='coerce'); t2 = t2.dropna(subset=['A'])
bas_hot = img2.select('hot').reduceRegion(ee.Reducer.mean(), basin, 90, crs=CRS, maxPixels=1e13, tileScale=8).getInfo()['hot']
bas_A = float(res_clim.loc['Aact_RUSLE', 'A media (t/ha/año)'])
print(f'A entorno PAM {t2.A.mean():.1f} vs cuenca {bas_A:.1f} | fracción hotspot {t2.hot.mean():.3f} vs {bas_hot:.3f}')
print(t2.groupby('provincia').agg(n=('A','size'), A=('A','mean'), hot=('hot','mean')).sort_values('n', ascending=False).round(3).to_string())
print(t2.groupby('exunidad').agg(n=('A','size'), A=('A','mean'), hot=('hot','mean')).query('n>=5').sort_values('hot', ascending=False).head(10).round(3).to_string())
t2.to_csv('tabla_pasivos_erosion_RUSLE.csv', index=False)
# Validación: rendimiento medido en la estación Santa (Morera et al., 2013): 779 ± 162 t/km²/año
for b in ['Aact_USLE', 'Aact_RUSLE', 'Aact_RUSLE_MB']:
    y = res_clim.loc[b, 'Pérdida total (Mt/año)'] * 1e6 * SDR / basin_area
    print(f'{b}: {y:,.0f} t/km²/año -> {y/779:.1f} veces lo medido | SDR que ajusta: {779*basin_area/(res_clim.loc[b, "Pérdida total (Mt/año)"]*1e6):.3f}')

## 15. Contraste de CHIRPS con pluviómetros y del área con La Balsa

In [ ]:
# Contraste de CHIRPS con pluviómetros del Alto Santa (IDR, INRENA-UGRH, SENAMHI y EGENOR, 2003)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'rasterio'], check=False)
import rasterio
st = pd.DataFrame({'estacion': ['Cahuish', 'Querococha', 'Ticapampa', 'Huarazquillcay'],
    'lat': [-9.6847, -9.7294, -9.7525, -9.4928], 'lon': [-77.2547, -77.3325, -77.4422, -77.5136],
    'alt': [4550, 4087, 3480, 3050], 'P_obs': [989.2, 935.5, 744.1, 688.8]})
vals = {k: [] for k in st.estacion}
for y in range(1995, 2025):
    with rasterio.open(f'/vsicurl/https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_annual/tifs/chirps-v2.0.{y}.tif') as ds:
        for k, v in zip(st.estacion, ds.sample(list(zip(st.lon, st.lat)))): vals[k].append(float(v[0]))
st['P_CHIRPS'] = [np.mean(vals[k]) for k in st.estacion]
st['sesgo_%'] = 100 * (st.P_CHIRPS - st.P_obs) / st.P_obs
print(st.round(1).to_string()); print(f"Sesgo medio {st['sesgo_%'].mean():.1f} % | MAE {np.mean(np.abs(st.P_CHIRPS - st.P_obs)):.1f} mm")
# Área publicada de La Balsa: 4 768 km² (Baraer et al., 2012)
print(f'Diferencia de área: {100*(basin_area-4768)/4768:.1f} %')